In [0]:
%python
# %pip install xgboost
# %pip install xgboost shap
%pip install -r ../../requirements.txt  -qqq

In [0]:
%python
# ==============================================================================
# Pipeline Databricks: Validação das Hipóteses H1 e H2 via XGBoost e SHAP
# Corrigido para o schema real de credito_prd.silver.give_me_some_credit
# ==============================================================================

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier
import shap

# 1. Leitura direta da tabela Silver no Unity Catalog
source_table = "credito_prd.silver.give_me_some_credit"
print(f"Lendo dados de: {source_table}")

spark_df = spark.table(source_table)
df = spark_df.toPandas()

# 2. Definição da Target e Features
# Nomes conforme DESCRIBE TABLE EXTENDED da Silver real
target_col = "target_dlq_2yrs"

# Colunas que NÃO são features de negócio: chave, metadado técnico e colunas
# internas do Delta/Autoloader (começam com "_" ou são de sistema)
cols_to_ignore = [
    target_col,
    "customer_id",
    "_ingestion_timestamp",
    "_source_file",
    "_silver_processed_at",
    "_metadata",
    "_object_metadata",
]
feature_cols = [col for col in df.columns if col not in cols_to_ignore]

print(f"Features utilizadas ({len(feature_cols)}): {feature_cols}")

X = df[feature_cols]
y = df[target_col].astype(int)

# 3. Split Estratificado (Treino / Teste) antes de qualquer transformação
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# 4. Construção do ColumnTransformer
# monthly_income e num_dependents são as colunas conhecidas com nulos no dataset original
cols_with_nulls = [c for c in ["monthly_income", "num_dependents"] if c in feature_cols]
other_numeric_cols = [c for c in feature_cols if c not in cols_with_nulls]

preprocessor = ColumnTransformer(
    transformers=[
        ("impute_median", SimpleImputer(strategy="median"), cols_with_nulls),
        ("passthrough_numeric", SimpleImputer(strategy="median"), other_numeric_cols),
    ],
    remainder="drop",
    verbose_feature_names_out=False,
)

# Ajuste do pré-processador exclusivamente no conjunto de treino
X_train_prep = preprocessor.fit_transform(X_train)
X_test_prep = preprocessor.transform(X_test)

transformed_feature_names = list(preprocessor.get_feature_names_out())
X_train_df = pd.DataFrame(X_train_prep, columns=transformed_feature_names, index=X_train.index)
X_test_df = pd.DataFrame(X_test_prep, columns=transformed_feature_names, index=X_test.index)

# 5. Cálculo do balanceamento de classes
neg_count = (y_train == 0).sum()
pos_count = (y_train == 1).sum()
scale_pos_weight_value = float(neg_count / pos_count)

print(f"Total Negativos: {neg_count} | Total Positivos: {pos_count}")
print(f"scale_pos_weight configurado: {scale_pos_weight_value:.2f}")

# 6. Treinamento do Modelo Principal: XGBoost
xgb_model = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=4,
    scale_pos_weight=scale_pos_weight_value,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric="auc",
)
xgb_model.fit(X_train_df, y_train)

# 7. Treinamento do Modelo Baseline: Regressão Logística com Padronização
logreg_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("logreg", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42)),
])
logreg_pipeline.fit(X_train_df, y_train)

# 8. Avaliação ROC-AUC
y_pred_xgb = xgb_model.predict_proba(X_test_df)[:, 1]
y_pred_logreg = logreg_pipeline.predict_proba(X_test_df)[:, 1]

print(f"\nROC-AUC XGBoost: {roc_auc_score(y_test, y_pred_xgb):.4f}")
print(f"ROC-AUC Regressão Logística: {roc_auc_score(y_test, y_pred_logreg):.4f}")

# 9. Cálculo dos valores de SHAP
explainer = shap.TreeExplainer(xgb_model)
shap_sample = X_test_df.sample(n=min(2000, len(X_test_df)), random_state=42)
shap_values = explainer(shap_sample)

# 10. Importância de Features e Coeficientes
mean_abs_shap = np.abs(shap_values.values).mean(axis=0)
shap_importance_df = pd.DataFrame({
    "feature": transformed_feature_names,
    "mean_abs_shap": mean_abs_shap,
}).sort_values(by="mean_abs_shap", ascending=False).reset_index(drop=True)

logreg_coefs = pd.DataFrame({
    "feature": transformed_feature_names,
    "coeficiente_logreg": logreg_pipeline.named_steps["logreg"].coef_[0],
}).sort_values(by="coeficiente_logreg", ascending=False).reset_index(drop=True)

print("\n--- Ranking de Importância SHAP (Top Features) ---")
print(shap_importance_df)

print("\n--- Coeficientes da Regressão Logística ---")
print(logreg_coefs)

# -------------------------------------------------------------
# Validação da Hipótese H1: revolving_utilization
# -------------------------------------------------------------
FEATURE_H1 = "revolving_utilization"

pos_util = int(shap_importance_df[shap_importance_df["feature"] == FEATURE_H1].index[0] + 1)
util_shap_val = float(
    shap_importance_df.loc[shap_importance_df["feature"] == FEATURE_H1, "mean_abs_shap"].values[0]
)
corr_util = float(
    np.corrcoef(shap_sample[FEATURE_H1], shap_values[:, FEATURE_H1].values)[0, 1]
)

# Margem mínima de correlação para considerar o efeito relevante, não só o sinal
CORR_THRESHOLD = 0.05

print("\n================== VALIDAÇÃO H1 ==================")
print(f"Variável: {FEATURE_H1}")
print(f"Posição no ranking SHAP: {pos_util}º lugar (Impacto médio: {util_shap_val:.4f})")
print(f"Correlação Feature vs SHAP: {corr_util:.4f}")
h1_status = "Confirmada" if corr_util > CORR_THRESHOLD else "Contrariada / Inconclusiva"
print(f"Conclusão H1: {h1_status}")

# -------------------------------------------------------------
# Validação da Hipótese H2: debt_ratio vs. age
# -------------------------------------------------------------
FEATURE_H2_A = "debt_ratio"
FEATURE_H2_B = "age"

debtratio_shap = float(
    shap_importance_df.loc[shap_importance_df["feature"] == FEATURE_H2_A, "mean_abs_shap"].values[0]
)
age_shap = float(
    shap_importance_df.loc[shap_importance_df["feature"] == FEATURE_H2_B, "mean_abs_shap"].values[0]
)

debtratio_pos = int(shap_importance_df[shap_importance_df["feature"] == FEATURE_H2_A].index[0] + 1)
age_pos = int(shap_importance_df[shap_importance_df["feature"] == FEATURE_H2_B].index[0] + 1)

debtratio_coef = float(
    logreg_coefs.loc[logreg_coefs["feature"] == FEATURE_H2_A, "coeficiente_logreg"].values[0]
)
age_coef = float(
    logreg_coefs.loc[logreg_coefs["feature"] == FEATURE_H2_B, "coeficiente_logreg"].values[0]
)

# Diferença relativa mínima para considerar debt_ratio "dominante" sobre age,
# em vez de só checar debtratio_shap > age_shap (margem zero)
RELATIVE_MARGIN = 0.10  # debt_ratio precisa ser pelo menos 10% maior que age

print("\n================== VALIDAÇÃO H2 ==================")
print(f"debt_ratio -> Ranking SHAP: {debtratio_pos}º (|SHAP|: {debtratio_shap:.4f}) | Coef RegLog: {debtratio_coef:.4f}")
print(f"age        -> Ranking SHAP: {age_pos}º (|SHAP|: {age_shap:.4f}) | Coef RegLog: {age_coef:.4f}")

relative_diff = (debtratio_shap - age_shap) / age_shap if age_shap > 0 else float("inf")
h2_status = "Confirmada" if relative_diff > RELATIVE_MARGIN else "Contrariada / Inconclusiva"
print(f"Diferença relativa (debt_ratio vs age): {relative_diff:.2%}")
print(f"Conclusão H2: {h2_status}")

# 11. Persistência dos Resultados na Camada Gold
gold_hypotheses_df = pd.DataFrame([
    {
        "hipotese": "H1",
        "descricao": "Clientes com maior revolving_utilization têm maior probabilidade de inadimplência",
        "metrica_shap": util_shap_val,
        "ranking_shap": pos_util,
        "correlacao_feature_shap": corr_util,
        "status": h1_status,
        "evidencia": f"Correlação SHAP-Feature de {corr_util:.4f} (limiar: {CORR_THRESHOLD}). Posição {pos_util}º no ranking global.",
    },
    {
        "hipotese": "H2",
        "descricao": "debt_ratio tem poder preditivo maior que age isoladamente",
        "metrica_shap": debtratio_shap,
        "ranking_shap": debtratio_pos,
        "correlacao_feature_shap": None,
        "status": h2_status,
        "evidencia": f"debt_ratio SHAP ({debtratio_shap:.4f}) vs age SHAP ({age_shap:.4f}). Diferença relativa: {relative_diff:.2%} (limiar: {RELATIVE_MARGIN:.0%}).",
    },
])

target_gold_table = "credito_prd.gold.gold_hypotheses_validation"
spark_gold_df = spark.createDataFrame(gold_hypotheses_df)
spark_gold_df.write.mode("overwrite").format("delta").saveAsTable(target_gold_table)

print(f"\nTabela Delta salva com sucesso no catálogo: {target_gold_table}")

In [0]:
-- %python
-- # prompt= https://gemini.google.com/app/76de07bcdd30d673

-- #            https://claude.ai/chat/bfce4019-d9fe-420f-9666-8bd14de3df42

-- # ==============================================================================
-- # Pipeline Databricks: Validação das Hipóteses H1 e H2 via XGBoost e SHAP
-- # Utilizando ColumnTransformer para Pré-processamento sem Data Leakage
-- # ==============================================================================

-- import numpy as np
-- import pandas as pd
-- from sklearn.model_selection import train_test_split
-- from sklearn.compose import ColumnTransformer
-- from sklearn.impute import SimpleImputer
-- from sklearn.preprocessing import StandardScaler
-- from sklearn.linear_model import LogisticRegression
-- from sklearn.pipeline import Pipeline
-- from sklearn.metrics import roc_auc_score
-- from xgboost import XGBClassifier
-- import shap

-- # 1. Leitura direta da tabela Silver no Databricks Unity Catalog
-- source_table = "credito_prd.silver.give_me_some_credit"
-- print(f"Lendo dados de: {source_table}")

-- spark_df = spark.table(source_table)
-- df = spark_df.toPandas()

-- # 2. Definição da Target e Features
-- target_col = "SeriousDlqin2yrs"
-- cols_to_ignore = [target_col, "id", "customer_id", "_c0", "index"]
-- feature_cols = [col for col in df.columns if col not in cols_to_ignore]

-- X = df[feature_cols]
-- y = df[target_col].astype(int)

-- # 3. Split Estratificado (Treino / Teste) antes de qualquer transformação
-- X_train, X_test, y_train, y_test = train_test_split(
--     X, y, test_size=0.20, random_state=42, stratify=y
-- )

-- # 4. Construção do ColumnTransformer
-- # Identifica colunas conhecidas com nulos no dataset e as demais colunas numéricas
-- cols_with_nulls = [c for c in ["MonthlyIncome", "NumberOfDependents"] if c in feature_cols]
-- other_numeric_cols = [c for c in feature_cols if c not in cols_with_nulls]

-- preprocessor = ColumnTransformer(
--     transformers=[
--         ("impute_median", SimpleImputer(strategy="median"), cols_with_nulls),
--         ("passthrough_numeric", SimpleImputer(strategy="median"), other_numeric_cols)
--     ],
--     remainder="drop",
--     verbose_feature_names_out=False
-- )

-- # Ajuste do pré-processador exclusivamente no conjunto de treino
-- X_train_prep = preprocessor.fit_transform(X_train)
-- X_test_prep = preprocessor.transform(X_test)

-- # Recuperação dos nomes das colunas após transformação para o SHAP
-- transformed_feature_names = list(preprocessor.get_feature_names_out())
-- X_train_df = pd.DataFrame(X_train_prep, columns=transformed_feature_names, index=X_train.index)
-- X_test_df = pd.DataFrame(X_test_prep, columns=transformed_feature_names, index=X_test.index)

-- # 5. Cálculo do balanceamento de classes
-- neg_count = (y_train == 0).sum()
-- pos_count = (y_train == 1).sum()
-- scale_pos_weight_value = float(neg_count / pos_count)

-- print(f"Total Negativos: {neg_count} | Total Positivos: {pos_count}")
-- print(f"scale_pos_weight configurado: {scale_pos_weight_value:.2f}")

-- # 6. Treinamento do Modelo Principal: XGBoost
-- xgb_model = XGBClassifier(
--     n_estimators=300,
--     learning_rate=0.05,
--     max_depth=4,
--     scale_pos_weight=scale_pos_weight_value,
--     subsample=0.8,
--     colsample_bytree=0.8,
--     random_state=42,
--     eval_metric="auc"
-- )
-- xgb_model.fit(X_train_df, y_train)

-- # 7. Treinamento do Modelo Baseline: Regressão Logística com Padronização
-- logreg_pipeline = Pipeline([
--     ("scaler", StandardScaler()),
--     ("logreg", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42))
-- ])
-- logreg_pipeline.fit(X_train_df, y_train)

-- # 8. Avaliação ROC-AUC
-- y_pred_xgb = xgb_model.predict_proba(X_test_df)[:, 1]
-- y_pred_logreg = logreg_pipeline.predict_proba(X_test_df)[:, 1]

-- print(f"\nROC-AUC XGBoost: {roc_auc_score(y_test, y_pred_xgb):.4f}")
-- print(f"ROC-AUC Regressão Logística: {roc_auc_score(y_test, y_pred_logreg):.4f}")

-- # 9. Cálculo dos valores de SHAP
-- explainer = shap.TreeExplainer(xgb_model)
-- shap_sample = X_test_df.sample(n=min(2000, len(X_test_df)), random_state=42)
-- shap_values = explainer(shap_sample)

-- # 10. Importância de Features e Coeficientes
-- mean_abs_shap = np.abs(shap_values.values).mean(axis=0)
-- shap_importance_df = pd.DataFrame({
--     "feature": transformed_feature_names,
--     "mean_abs_shap": mean_abs_shap
-- }).sort_values(by="mean_abs_shap", ascending=False).reset_index(drop=True)

-- logreg_coefs = pd.DataFrame({
--     "feature": transformed_feature_names,
--     "coeficiente_logreg": logreg_pipeline.named_steps["logreg"].coef_[0]
-- }).sort_values(by="coeficiente_logreg", ascending=False).reset_index(drop=True)

-- print("\n--- Ranking de Importância SHAP (Top Features) ---")
-- print(shap_importance_df)

-- print("\n--- Coeficientes da Regressão Logística ---")
-- print(logreg_coefs)

-- # -------------------------------------------------------------
-- # Validação da Hipótese H1: RevolvingUtilizationOfUnsecuredLines
-- # -------------------------------------------------------------
-- pos_util = int(shap_importance_df[shap_importance_df["feature"] == "RevolvingUtilizationOfUnsecuredLines"].index[0] + 1)
-- util_shap_val = float(shap_importance_df.loc[shap_importance_df["feature"] == "RevolvingUtilizationOfUnsecuredLines", "mean_abs_shap"].values[0])
-- corr_util = float(np.corrcoef(shap_sample["RevolvingUtilizationOfUnsecuredLines"], shap_values[:, "RevolvingUtilizationOfUnsecuredLines"].values)[0, 1])

-- print("\n================== VALIDAÇÃO H1 ==================")
-- print(f"Variável: RevolvingUtilizationOfUnsecuredLines")
-- print(f"Posição no ranking SHAP: {pos_util}º lugar (Impacto médio: {util_shap_val:.4f})")
-- print(f"Correlação Feature vs SHAP: {corr_util:.4f}")
-- h1_status = "Confirmada" if corr_util > 0 else "Contrariada"
-- print(f"Conclusão H1: {h1_status}")

-- # -------------------------------------------------------------
-- # Validação da Hipótese H2: DebtRatio vs. age
-- # -------------------------------------------------------------
-- debtratio_shap = float(shap_importance_df.loc[shap_importance_df["feature"] == "DebtRatio", "mean_abs_shap"].values[0])
-- age_shap = float(shap_importance_df.loc[shap_importance_df["feature"] == "age", "mean_abs_shap"].values[0])

-- debtratio_pos = int(shap_importance_df[shap_importance_df["feature"] == "DebtRatio"].index[0] + 1)
-- age_pos = int(shap_importance_df[shap_importance_df["feature"] == "age"].index[0] + 1)

-- debtratio_coef = float(logreg_coefs.loc[logreg_coefs["feature"] == "DebtRatio", "coeficiente_logreg"].values[0])
-- age_coef = float(logreg_coefs.loc[logreg_coefs["feature"] == "age", "coeficiente_logreg"].values[0])

-- print("\n================== VALIDAÇÃO H2 ==================")
-- print(f"DebtRatio -> Ranking SHAP: {debtratio_pos}º (|SHAP|: {debtratio_shap:.4f}) | Coef RegLog: {debtratio_coef:.4f}")
-- print(f"age       -> Ranking SHAP: {age_pos}º (|SHAP|: {age_shap:.4f}) | Coef RegLog: {age_coef:.4f}")
-- h2_status = "Confirmada" if debtratio_shap > age_shap else "Contrariada"
-- print(f"Conclusão H2: {h2_status}")

-- # 11. Persistência dos Resultados na Camada Gold
-- gold_hypotheses_df = pd.DataFrame([
--     {
--         "hipotese": "H1",
--         "descricao": "Clientes com maior RevolvingUtilization têm maior probabilidade de inadimplência",
--         "metrica_shap": util_shap_val,
--         "ranking_shap": pos_util,
--         "status": h1_status,
--         "evidencia": f"Correlação SHAP-Feature de {corr_util:.4f}. Variável posicionada em {pos_util}º no ranking global."
--     },
--     {
--         "hipotese": "H2",
--         "descricao": "DebtRatio tem poder preditivo maior que a idade isoladamente",
--         "metrica_shap": debtratio_shap,
--         "ranking_shap": debtratio_pos,
--         "status": h2_status,
--         "evidencia": f"DebtRatio SHAP ({debtratio_shap:.4f}) vs Age SHAP ({age_shap:.4f})."
--     }
-- ])

-- target_gold_table = "credito_prd.gold.gold_hypotheses_validation"
-- spark_gold_df = spark.createDataFrame(gold_hypotheses_df)
-- spark_gold_df.write.mode("overwrite").format("delta").saveAsTable(target_gold_table)

-- print(f"\nTabela Delta salva com sucesso no catálogo: {target_gold_table}")